In [1]:
# Cell 1 — Load env and init client
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION")
)

print("Client ready ✅")

print(os.getenv("AZURE_OPENAI_ENDPOINT"))

print(os.getenv("AZURE_OPENAI_KEY"))

print(os.getenv("AZURE_OPENAI_API_VERSION"))



Client ready ✅
https://open-ai-wt-poc.openai.azure.com/
    
2025-04-01-preview


In [2]:
from pydantic import BaseModel
from typing import Optional

class DailyReport(BaseModel):
    Vendor: str
    Date: str          # YYYY-MM-DD
    TotalWorkers: int
    TotalWorkHr: float

In [3]:
import base64

def image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as f:      # rb = read as bytes
        image_bytes = f.read()
    return base64.b64encode(image_bytes).decode("utf-8")

# --- point this to your image ---
IMAGE_PATH = "img1.png"   # change this to your actual filename

image_b64 = image_to_base64(IMAGE_PATH)

print(f"Image loaded ✅ — base64 length: {len(image_b64)} chars")

Image loaded ✅ — base64 length: 154520 chars


In [6]:
response = client.beta.chat.completions.parse(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
    messages=[
        {"role": "system", "content": "You extract structured data from contractor daily work report images."},
        {"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}", "detail": "high"}},
            {"type": "text", "text": """Extract from this contractor daily work report:
- Vendor: contractor company name who gives service to Whiting Turner Company
- Date: date on the form as YYYY-MM-DD. 
  IMPORTANT: year is always 2024 or 2025, never 1900s
- TotalWorkers: count of individual worker rows listed
- TotalWorkHr: use the printed Total Manhours field if filled.
  If blank, ADD UP each worker's individual hours"""}
        ]}
    ],
    response_format=DailyReport,  # single source of truth
    temperature=0
)

# refusal check — production habit
if response.choices[0].message.refusal:
    print("Model refused:", response.choices[0].message.refusal)
else:
    result = response.choices[0].message.parsed  # typed DailyReport object
    print(result)
    print(f"\nTokens — input: {response.usage.prompt_tokens}, output: {response.usage.completion_tokens}")

Vendor='CSC' Date='2025-02-17' TotalWorkers=4 TotalWorkHr=32.0

Tokens — input: 798, output: 26


In [10]:
import base64

def image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as f:      # rb = read as bytes
        image_bytes = f.read()
    return base64.b64encode(image_bytes).decode("utf-8")

# --- point this to your image ---
IMAGE_PATH = "img2.jpg"   # change this to your actual filename

image_b64 = image_to_base64(IMAGE_PATH)

print(f"Image loaded ✅ — base64 length: {len(image_b64)} chars")


response = client.beta.chat.completions.parse(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
    messages=[
        {"role": "system", "content": "You extract structured data from contractor daily work report images."},
        {"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}", "detail": "high"}},
            {"type": "text", "text": """Extract from this contractor daily work report:
- Vendor: contractor company name who gives service to Whiting Turner Company
- Date: date on the form as YYYY-MM-DD. 
  IMPORTANT: year is always 2024 or 2025, never 1900s
- TotalWorkers: count of individual worker rows listed
- TotalWorkHr: use the printed Total Manhours field if filled.
  If blank, ADD UP each worker's individual hours"""}
        ]}
    ],
    response_format=DailyReport,  # single source of truth
    temperature=0
)

# refusal check — production habit
if response.choices[0].message.refusal:
    print("Model refused:", response.choices[0].message.refusal)
else:
    result = response.choices[0].message.parsed  # typed DailyReport object
    print(result)
    print(f"\nTokens — input: {response.usage.prompt_tokens}, output: {response.usage.completion_tokens}")







Image loaded ✅ — base64 length: 125984 chars
Vendor='H.T.K masonry' Date='2024-12-17' TotalWorkers=2 TotalWorkHr=32.0

Tokens — input: 864, output: 29


In [11]:
class DailyReport(BaseModel):
    reasoning: str      # model explains its logic BEFORE giving values
    Vendor: str
    Date: str           # YYYY-MM-DD
    TotalWorkers: int
    TotalWorkHr: float

In [22]:
import base64

def image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as f:      # rb = read as bytes
        image_bytes = f.read()
    return base64.b64encode(image_bytes).decode("utf-8")

# --- point this to your image ---
IMAGE_PATH = "img2.jpg"   # change this to your actual filename

image_b64 = image_to_base64(IMAGE_PATH)

print(f"Image loaded ✅ — base64 length: {len(image_b64)} chars")


response = client.beta.chat.completions.parse(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
    messages=[
        {"role": "system", "content": "You extract structured data from contractor daily work report images."},
        {"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}", "detail": "high"}},
            {"type": "text", "text": """Extract from this contractor daily work report.

First explain your reasoning for each field, then extract:
- Vendor: contractor company name
- Date: YYYY-MM-DD format. Year is always 2024 or 2025
- TotalWorkers: SUM of the Number column across all rows
  (each row has Classification, Number, Hours - add up the Number column)
- TotalWorkHr: use printed Total Manhours if filled, else sum hours×workers per row
"""}
        ]}
    ],
    response_format=DailyReport,  # single source of truth
    temperature=0
)

print(response)
print()
print(response.choices[0])
print()
print(response.choices[0].message)
print()
print()
print(response.choices[0].message.parsed) 

print("respons:")


# refusal check — production habit
if response.choices[0].message.refusal:
    print("Model refused:", response.choices[0].message.refusal)
else:
    result = response.choices[0].message.parsed  # typed DailyReport object
    print(result)
    print(f"\nTokens — input: {response.usage.prompt_tokens}, output: {response.usage.completion_tokens}")







Image loaded ✅ — base64 length: 125984 chars
ParsedChatCompletion[TypeVar](id='chatcmpl-DSMNypHFcrBbIhyrBvHr6Jbl8oGSc', choices=[ParsedChoice[TypeVar](finish_reason='stop', index=0, logprobs=None, message=ParsedChatCompletionMessage[TypeVar](content='{"reasoning":"The vendor is the contractor company name, which is handwritten as \'H.T.K masonry\'. The date is given as \'12-17-24\', which corresponds to 2024-12-17 in YYYY-MM-DD format. The total number of workers is the sum of the \'Number\' column: 2 (Foreman) + 2 (Bricklayers) = 4. The total manhours worked is printed as 32, so we use that value for TotalWorkHr.","Vendor":"H.T.K masonry","Date":"2024-12-17","TotalWorkers":4,"TotalWorkHr":32}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None, parsed=DailyReport(reasoning="The vendor is the contractor company name, which is handwritten as 'H.T.K masonry'. The date is given as '12-17-24', which corresponds to 2024-12-17 in YYYY-MM-DD forma

In [20]:
print(response)
print()
print(response.choices[0])
print()
print(response.choices[0].message)
print()
print()
print(response.choices[0].message.parsed) 

ParsedChatCompletion[TypeVar](id='chatcmpl-DSLWfOomLn0Hp2FoEwXp35YeZmLI7', choices=[ParsedChoice[TypeVar](finish_reason='stop', index=0, logprobs=None, message=ParsedChatCompletionMessage[TypeVar](content='{"reasoning":"The vendor is the contractor company name, which is handwritten as \'H.T.K masonry\'. The date is given as 12/17/24, which corresponds to 2024-12-17 in YYYY-MM-DD format. The total workers are the sum of the \'Number\' column: 2 (Foreman) + 2 (Bricklayers) = 4. The total manhours worked is printed as 32, so we use that value for TotalWorkHr.","Vendor":"H.T.K masonry","Date":"2024-12-17","TotalWorkers":4,"TotalWorkHr":32}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None, parsed=DailyReport(reasoning="The vendor is the contractor company name, which is handwritten as 'H.T.K masonry'. The date is given as 12/17/24, which corresponds to 2024-12-17 in YYYY-MM-DD format. The total workers are the sum of the 'Number' column: 2 (

In [21]:
result = response.choices[0].message.parsed

# Option 1 — Python dict (for passing to UI/API)
result_dict = result.model_dump()
print(result_dict)
# {'reasoning': '...', 'Vendor': 'H.T.K masonry', 'Date': '2024-12-17', ...}

# Option 2 — JSON string (for HTTP response / frontend)
result_json = result.model_dump_json()
print(result_json)
# '{"reasoning":"...","Vendor":"H.T.K masonry","Date":"2024-12-17",...}'

# Option 3 — exclude reasoning before sending to UI
result_clean = result.model_dump(exclude={"reasoning"})
print(result_clean)
# {'Vendor': 'H.T.K masonry', 'Date': '2024-12-17', 'TotalWorkers': 4, 'TotalWorkHr': 32.0}

{'reasoning': "The vendor is the contractor company name, which is handwritten as 'H.T.K masonry'. The date is given as 12/17/24, which corresponds to 2024-12-17 in YYYY-MM-DD format. The total workers are the sum of the 'Number' column: 2 (Foreman) + 2 (Bricklayers) = 4. The total manhours worked is printed as 32, so we use that value for TotalWorkHr.", 'Vendor': 'H.T.K masonry', 'Date': '2024-12-17', 'TotalWorkers': 4, 'TotalWorkHr': 32.0}
{"reasoning":"The vendor is the contractor company name, which is handwritten as 'H.T.K masonry'. The date is given as 12/17/24, which corresponds to 2024-12-17 in YYYY-MM-DD format. The total workers are the sum of the 'Number' column: 2 (Foreman) + 2 (Bricklayers) = 4. The total manhours worked is printed as 32, so we use that value for TotalWorkHr.","Vendor":"H.T.K masonry","Date":"2024-12-17","TotalWorkers":4,"TotalWorkHr":32.0}
{'Vendor': 'H.T.K masonry', 'Date': '2024-12-17', 'TotalWorkers': 4, 'TotalWorkHr': 32.0}


In [ ]:
import base64

def image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as f:      # rb = read as bytes
        image_bytes = f.read()
    return base64.b64encode(image_bytes).decode("utf-8")

# --- point this to your image ---
IMAGE_PATH = "img2.jpg"   # change this to your actual filename

image_b64 = image_to_base64(IMAGE_PATH)

print(f"Image loaded ✅ — base64 length: {len(image_b64)} chars")


response = client.beta.chat.completions.parse(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
    messages=[
        {"role": "system", "content": "You extract structured data from contractor daily work report images."},
        {"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}", "detail": "high"}},
            {"type": "text", "text": """Extract from this contractor daily work report.

First explain your reasoning for each field, then extract:
- Vendor: contractor company name
- Date: YYYY-MM-DD format. Year is always 2024 or 2025
- TotalWorkers: SUM of the Number column across all rows
  (each row has Classification, Number, Hours - add up the Number column)
- TotalWorkHr: use printed Total Manhours if filled, else sum hours×workers per row
"""}
        ]}
    ],
    response_format=DailyReport,  # single source of truth
    temperature=0
)

print(response)
print()
print(response.choices[0])
print()
print(response.choices[0].message)
print()
print()
print(response.choices[0].message.parsed) 

print("respons:")


# refusal check — production habit
if response.choices[0].message.refusal:
    print("Model refused:", response.choices[0].message.refusal)
else:
    result = response.choices[0].message.parsed  # typed DailyReport object
    print(result)
    print(f"\nTokens — input: {response.usage.prompt_tokens}, output: {response.usage.completion_tokens}")







In [24]:
import os
import json
import base64
from pathlib import Path

# --- same as before ---
def image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def extract_from_image(image_path: str):
    """
    Same LLM call you already have — just wrapped in a function.
    Returns a DailyReport object or None if something goes wrong.
    """
    image_b64 = image_to_base64(image_path)

    # Detect image type from extension
    ext = Path(image_path).suffix.lower()
    mime = "image/png" if ext == ".png" else "image/jpeg"

    try:
        response = client.beta.chat.completions.parse(
            model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
            messages=[
                {"role": "system", "content": "You extract structured data from contractor daily work report images."},
                {"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{image_b64}", "detail": "high"}},
                    {"type": "text", "text": """Extract from this contractor daily work report.
- Vendor: contractor company name
- Date: YYYY-MM-DD format. Year is always 2024 or 2025
- TotalWorkers: SUM of the Number column across all rows
- TotalWorkHr: use printed Total Manhours if filled, else sum hours×workers per row
"""}
                ]}
            ],
            response_format=DailyReport,
            temperature=0
        )

        if response.choices[0].message.refusal:
            print(f"  ⚠️  Model refused for {image_path}")
            return None

        return response.choices[0].message.parsed

    except Exception as e:
        print(f"  ❌ Error on {image_path}: {e}")
        return None


# -------------------------------------------------------
# BATCH LOOP
# -------------------------------------------------------

IMGS_FOLDER = "imgs"
OUTPUT_FILE = "results.jsonl"   # one JSON object per line

# Build sorted list: img1, img2 ... img13
image_files = sorted(
    Path(IMGS_FOLDER).glob("img*.png"),  # change to img*.png if needed
    key=lambda p: int(p.stem.replace("img", ""))  # sort numerically
)

print(f"Found {len(image_files)} images\n")

with open(OUTPUT_FILE, "w") as out_f:
    for img_path in image_files:
        print(f"Processing {img_path.name} ...")

        result = extract_from_image(str(img_path))

        if result:
            # Add source filename so you know which image produced which result
            row = result.model_dump()
            row["source_file"] = img_path.name

            out_f.write(json.dumps(row) + "\n")
            out_f.flush()  # write to disk immediately — safe if crash mid-loop

            print(f"  ✅ {result.Vendor} | {result.Date} | Workers: {result.TotalWorkers}")
        else:
            print(f"  ⚠️  Skipped {img_path.name}")

print(f"\nDone. Results saved to {OUTPUT_FILE}")

Found 13 images

Processing img1.png ...
  ✅ The Whiting-Turner Contracting Company | 2025-02-17 | Workers: 4
Processing img2.png ...
  ✅ H.T.K masonry | 2024-12-17 | Workers: 4
Processing img3.png ...
  ✅ Corrado | 2024-10-02 | Workers: 4
Processing img4.png ...
  ✅ Nickle Electrical Companies | 2025-01-23 | Workers: 5
Processing img5.png ...
  ✅ WOLF Contractors, Inc. | 2024-03-12 | Workers: 6
Processing img6.png ...
  ✅ Tirone Electric Data | 2025-08-15 | Workers: 6
Processing img7.png ...
  ✅ ELECTRICO INC. | 2025-12-19 | Workers: 8
Processing img8.png ...
  ✅ Md fab | 2025-09-17 | Workers: 4
Processing img9.png ...
  ✅ THE WHITING-TURNER CONTRACTING COMPANY |  | Workers: 3
Processing img10.png ...
  ✅ THE WHITING-TURNER CONTRACTING COMPANY | 2025-02-25 | Workers: 3
Processing img11.png ...
  ✅ William R. Nash Mechanical Contractors | 2024-08-03 | Workers: 3
Processing img12.png ...
  ✅ Prescision concrete | 2024-03-20 | Workers: 7
Processing img13.png ...
  ✅ S.M.I. | 2024-12-06 |